# Limpieza y preparación de datasets

Este notebook corresponde a la fase de limpieza y preparación de datos del proyecto.

El objetivo principal es dejar preparados los datasets necesarios para que el resto del equipo pueda realizar el análisis exploratorio, las visualizaciones y la interpretación de resultados.

En esta fase se trabaja con tres datasets:

- `df_jobs`: dataset de ofertas de empleo.
- `df_tecno`: dataset de ofertas tecnológicas.
- `df_stack`: dataset de Stack Overflow con información sobre tecnologías usadas y demandadas.

Al final del proceso se generan varios archivos limpios en la carpeta `data/clean/`.



In [5]:
import os
from pathlib import Path

# Ruta base del proyecto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_CLEAN = PROJECT_ROOT / "data" / "clean"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

print("Carpeta actual:", Path.cwd())
print("Raíz del proyecto:", PROJECT_ROOT)
print("Carpeta raw:", DATA_RAW)
print("Carpeta clean:", DATA_CLEAN)

Carpeta actual: c:\Users\User\OneDrive\Documentos\BOOTCAMP IA\Proyecto1-modulo2\proyecto-eda-roles-datos\notebooks
Raíz del proyecto: c:\Users\User\OneDrive\Documentos\BOOTCAMP IA\Proyecto1-modulo2\proyecto-eda-roles-datos
Carpeta raw: c:\Users\User\OneDrive\Documentos\BOOTCAMP IA\Proyecto1-modulo2\proyecto-eda-roles-datos\data\raw
Carpeta clean: c:\Users\User\OneDrive\Documentos\BOOTCAMP IA\Proyecto1-modulo2\proyecto-eda-roles-datos\data\clean


In [6]:
print("Archivos en data/raw:")
for file in DATA_RAW.iterdir():
    print(file.name)

Archivos en data/raw:
.gitkeep
data_science_job_posts_2025.csv
stackoverflow_2025_results.csv
tecnoempleo_spain_2026.csv


## 1. Imports y configuración inicial

En esta sección se importan las librerías necesarias para trabajar con los datos.

También se configuran algunas opciones de visualización de pandas para poder ver más columnas y filas en el notebook.

Además, se crea la carpeta `data/clean/`, donde se guardarán los datasets limpios generados durante el proceso.

In [7]:
import pandas as pd
import numpy as np
import os
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

os.makedirs("data/clean", exist_ok=True)

## 2. Funciones generales de limpieza

Antes de limpiar cada dataset, se definen funciones reutilizables.

Estas funciones permiten:

- Normalizar los nombres de las columnas.
- Limpiar valores de texto.
- Eliminar espacios innecesarios.
- Convertir valores vacíos o textos como `"nan"`, `"none"` o `"null"` en valores nulos reales.

De esta forma se evita repetir código y se aplica una limpieza homogénea a los distintos datasets.

In [8]:
def normalizar_nombre_columnas(df):
    df = df.copy()
    
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("á", "a", regex=False)
        .str.replace("é", "e", regex=False)
        .str.replace("í", "i", regex=False)
        .str.replace("ó", "o", regex=False)
        .str.replace("ú", "u", regex=False)
    )
    
    return df


def limpiar_texto(valor):
    if pd.isna(valor):
        return np.nan
    
    valor = str(valor).strip()
    
    if valor == "" or valor.lower() in ["nan", "none", "null"]:
        return np.nan
    
    return valor


def limpiar_columnas_texto(df):
    df = df.copy()
    
    columnas_texto = df.select_dtypes(include=["object", "string"]).columns
    
    for col in columnas_texto:
        df[col] = df[col].apply(limpiar_texto)
    
    return df

## 3. Carga de datasets

En esta sección se cargan los tres datasets principales del proyecto.

Los datasets utilizados son:

| Dataset | Descripción |
|---|---|
| `df_jobs` | Contiene ofertas de empleo con información como puesto, empresa, ubicación, salario, industria y habilidades. |
| `df_tecno` | Contiene ofertas tecnológicas con información sobre título, empresa, salario, ubicación, tipo de trabajo, fecha y enlace. |
| `df_stack` | Contiene respuestas de una encuesta tecnológica, incluyendo tecnologías usadas, tecnologías deseadas y datos sobre perfiles profesionales. |

In [9]:
import pandas as pd
import numpy as np
#cargamos los csv
pd.set_option("display.max_columns", None)

df_jobs = pd.read_csv(DATA_RAW / "data_science_job_posts_2025.csv")
df_tecno = pd.read_csv(DATA_RAW / "tecnoempleo_spain_2026.csv")
df_stack = pd.read_csv(DATA_RAW / "stackoverflow_2025_results.csv")

print("Data Science Jobs:", df_jobs.shape)
print("TecnoEmpleo España:", df_tecno.shape)
print("Stack Overflow:", df_stack.shape)

C:\Users\User\AppData\Local\Temp\ipykernel_22736\4227587284.py:8: DtypeWarning: Columns (0: JobSatPoints_15_TEXT, 1: DatabaseHaveEntry, 2: DevEnvHaveEntry, 3: SOTagsHaveEntry, 4: SOTagsWant Entry, 5: OfficeStackWantEntry, 6: CommPlatformHaveEntr, 7: CommPlatformWantEntr, 8: SO_Actions_15_TEXT, 9: AIAgentOrchestration, 10: AIAgentObsWrite) have mixed types. Specify dtype option on import or set low_memory=False.
  df_stack = pd.read_csv(DATA_RAW / "stackoverflow_2025_results.csv")


Data Science Jobs: (944, 13)
TecnoEmpleo España: (600, 7)
Stack Overflow: (49191, 172)


## 4. Comprobación inicial de los datasets

Antes de aplicar la limpieza, se realiza una comprobación inicial de cada dataset.

En esta revisión se observa:

- El tamaño del dataset.
- Las primeras filas.
- Los nombres de las columnas.
- Los tipos de datos.
- La presencia de valores nulos.
- La existencia de duplicados.

Esta comprobación permite entender el estado inicial de los datos antes de modificarlos:

## 5. Limpieza de `df_jobs`

En esta sección se limpia el dataset de ofertas de empleo.

Las acciones realizadas son:

1. Crear una copia del dataset original para no modificarlo directamente.
2. Normalizar los nombres de las columnas.
3. Limpiar columnas de texto.
4. Eliminar duplicados.
5. Comprobar el tamaño del dataset antes y después de la limpieza.

El resultado se guarda posteriormente como `jobs_clean.csv`.

In [10]:
# Comprobación inicial de df_jobs
print("Tamaño de df_jobs:", df_jobs.shape)
df_jobs.head()


Tamaño de df_jobs: (944, 13)


,job_title,seniority_level,status,company,location,post_date,headquarter,industry,ownership,company_size,revenue,salary,skills
0,data scientist,senior,hybrid,company_003,"Grapevine, TX . Hybrid",17 days ago,"Bentonville, AR, US",Retail,Public,€352.44B,Public,"€100,472 - €200,938","['spark', 'r', 'python', 'scala', 'machine lea..."
1,data scientist,lead,hybrid,company_005,"Fort Worth, TX . Hybrid",15 days ago,"Detroit, MI, US",Manufacturing,Public,"155,030",€51.10B,"€118,733","['spark', 'r', 'python', 'sql', 'machine learn..."
2,data scientist,senior,on-site,company_007,"Austin, TX . Toronto, Ontario, Canada . Kirkla...",a month ago,"Redwood City, CA, US",Technology,Public,"25,930",€33.80B,"€94,987 - €159,559","['aws', 'git', 'python', 'docker', 'sql', 'mac..."
3,data scientist,senior,hybrid,company_008,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...",8 days ago,"San Jose, CA, US",Technology,Public,"34,690",€81.71B,"€112,797 - €194,402","['sql', 'r', 'python']"
4,data scientist,NaN,on-site,company_009,On-site,3 days ago,"Stamford, CT, US",Finance,Private,"1,800",Private,"€114,172 - €228,337",[]


In [11]:
df_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 944 entries, 0 to 943
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   job_title        941 non-null    str  
 1   seniority_level  884 non-null    str  
 2   status           688 non-null    str  
 3   company          944 non-null    str  
 4   location         942 non-null    str  
 5   post_date        944 non-null    str  
 6   headquarter      944 non-null    str  
 7   industry         944 non-null    str  
 8   ownership        897 non-null    str  
 9   company_size     944 non-null    str  
 10  revenue          929 non-null    str  
 11  salary           944 non-null    str  
 12  skills           944 non-null    str  
dtypes: str(13)
memory usage: 96.0 KB


In [12]:
df_jobs_clean = df_jobs.copy()

df_jobs_clean = normalizar_nombre_columnas(df_jobs_clean)
df_jobs_clean = limpiar_columnas_texto(df_jobs_clean)

duplicados_jobs = df_jobs_clean.duplicated().sum()
print(f"Duplicados encontrados en df_jobs: {duplicados_jobs}")

df_jobs_clean = df_jobs_clean.drop_duplicates()

print("Tamaño original:", df_jobs.shape)
print("Tamaño limpio:", df_jobs_clean.shape)

print(df_jobs_clean.columns.tolist())
df_jobs_clean.head()

Duplicados encontrados en df_jobs: 0
Tamaño original: (944, 13)
Tamaño limpio: (944, 13)
['job_title', 'seniority_level', 'status', 'company', 'location', 'post_date', 'headquarter', 'industry', 'ownership', 'company_size', 'revenue', 'salary', 'skills']


,job_title,seniority_level,status,company,location,post_date,headquarter,industry,ownership,company_size,revenue,salary,skills
0,data scientist,senior,hybrid,company_003,"Grapevine, TX . Hybrid",17 days ago,"Bentonville, AR, US",Retail,Public,€352.44B,Public,"€100,472 - €200,938","['spark', 'r', 'python', 'scala', 'machine lea..."
1,data scientist,lead,hybrid,company_005,"Fort Worth, TX . Hybrid",15 days ago,"Detroit, MI, US",Manufacturing,Public,"155,030",€51.10B,"€118,733","['spark', 'r', 'python', 'sql', 'machine learn..."
2,data scientist,senior,on-site,company_007,"Austin, TX . Toronto, Ontario, Canada . Kirkla...",a month ago,"Redwood City, CA, US",Technology,Public,"25,930",€33.80B,"€94,987 - €159,559","['aws', 'git', 'python', 'docker', 'sql', 'mac..."
3,data scientist,senior,hybrid,company_008,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...",8 days ago,"San Jose, CA, US",Technology,Public,"34,690",€81.71B,"€112,797 - €194,402","['sql', 'r', 'python']"
4,data scientist,NaN,on-site,company_009,On-site,3 days ago,"Stamford, CT, US",Finance,Private,"1,800",Private,"€114,172 - €228,337",[]


## 6. Limpieza de `df_tecno`

En esta sección se limpia el dataset de ofertas tecnológicas.

El objetivo es dejarlo preparado para poder analizar ofertas del sector tecnológico y compararlo con otros datos del proyecto.

Las acciones realizadas son:

1. Crear una copia del dataset original.
2. Normalizar nombres de columnas.
3. Limpiar textos.
4. Eliminar duplicados.
5. Revisar el tamaño final del dataset limpio.

El resultado se guarda posteriormente como `tecno_jobs_clean.csv`.

In [13]:
# Comprobación inicial de df_tecno
print("Tamaño de df_tecno:", df_tecno.shape)
df_tecno.head()

Tamaño de df_tecno: (600, 7)


,Titulo,Empresa,Salario,Ubicacion,Tipo de trabajo,Fecha de publicacion,Enlace
0,Urgente\t\t\t\t\t\t\t\t\t\t\n\r\n\t\t\t\t\t\t\...,NEXYRA,24.000€ - 36.000€,Alicante,Híbrido,03/04/2026,https://www.tecnoempleo.com/analista-programad...
1,Urgente\t\t\t\t\t\t\t\t\t\t\n\r\n\t\t\t\t\t\t\...,OKDIARIO,NaN,Madrid,Presencial,03/04/2026,https://www.tecnoempleo.com/tecnico-sistemas-s...
2,Urgente\t\t\t\t\t\t\t\t\t\t\n\r\n\t\t\t\t\t\t\...,EXDESIS,NaN,Barcelona,Híbrido,03/04/2026,https://www.tecnoempleo.com/senior-data-visual...
3,Urgente\t\t\t\t\t\t\t\t\t\t\n\r\n\t\t\t\t\t\t\...,Social You,45.000€ - 54.000€,Barcelona,Híbrido,03/04/2026,https://www.tecnoempleo.com/senior-fullstack-e...
4,Software Development Engineer,Airbus,NaN,Madrid,NaN,04/04/2026,https://www.tecnoempleo.com/software-developme...


In [14]:
df_tecno.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Titulo                600 non-null    str  
 1   Empresa               600 non-null    str  
 2   Salario               129 non-null    str  
 3   Ubicacion             419 non-null    str  
 4   Tipo de trabajo       387 non-null    str  
 5   Fecha de publicacion  600 non-null    str  
 6   Enlace                600 non-null    str  
dtypes: str(7)
memory usage: 32.9 KB


In [15]:
# ============================================================
# LIMPIEZA DE df_tecno
# ============================================================

df_tecno_clean = df_tecno.copy()

# 1. Normalizar nombres de columnas
df_tecno_clean = normalizar_nombre_columnas(df_tecno_clean)

# 2. Limpiar columnas de texto
df_tecno_clean = limpiar_columnas_texto(df_tecno_clean)

# 3. Eliminar duplicados
duplicados_tecno = df_tecno_clean.duplicated().sum()
print(f"Duplicados encontrados en df_tecno: {duplicados_tecno}")

df_tecno_clean = df_tecno_clean.drop_duplicates()

# 4. Revisar tamaño final
print("Tamaño original:", df_tecno.shape)
print("Tamaño limpio:", df_tecno_clean.shape)

# 5. Ver columnas y primeras filas
print(df_tecno_clean.columns.tolist())
df_tecno_clean.head()

Duplicados encontrados en df_tecno: 0
Tamaño original: (600, 7)
Tamaño limpio: (600, 7)
['titulo', 'empresa', 'salario', 'ubicacion', 'tipo_de_trabajo', 'fecha_de_publicacion', 'enlace']


,titulo,empresa,salario,ubicacion,tipo_de_trabajo,fecha_de_publicacion,enlace
0,Urgente\t\t\t\t\t\t\t\t\t\t\n\r\n\t\t\t\t\t\t\...,NEXYRA,24.000€ - 36.000€,Alicante,Híbrido,03/04/2026,https://www.tecnoempleo.com/analista-programad...
1,Urgente\t\t\t\t\t\t\t\t\t\t\n\r\n\t\t\t\t\t\t\...,OKDIARIO,NaN,Madrid,Presencial,03/04/2026,https://www.tecnoempleo.com/tecnico-sistemas-s...
2,Urgente\t\t\t\t\t\t\t\t\t\t\n\r\n\t\t\t\t\t\t\...,EXDESIS,NaN,Barcelona,Híbrido,03/04/2026,https://www.tecnoempleo.com/senior-data-visual...
3,Urgente\t\t\t\t\t\t\t\t\t\t\n\r\n\t\t\t\t\t\t\...,Social You,45.000€ - 54.000€,Barcelona,Híbrido,03/04/2026,https://www.tecnoempleo.com/senior-fullstack-e...
4,Software Development Engineer,Airbus,NaN,Madrid,NaN,04/04/2026,https://www.tecnoempleo.com/software-developme...


## 7. Limpieza de `df_stack`

El dataset `df_stack` contiene muchas columnas, pero para el objetivo del proyecto no es necesario utilizarlas todas.

Como el análisis se centra en identificar tecnologías más usadas y más demandadas, se seleccionan únicamente las columnas relacionadas con:

- Lenguajes de programación.
- Bases de datos.
- Plataformas o servicios cloud.
- Frameworks web.
- Entornos de desarrollo.
- Modelos o herramientas de inteligencia artificial.

A partir de estas columnas se crea un dataset reducido llamado `df_stack_tech`.

In [16]:
# Comprobación inicial de df_stack
print("Tamaño de df_stack:", df_stack.shape)
df_stack.head()

Tamaño de df_stack: (49191, 172)


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,AILearnHow,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,TechEndorse_1,TechEndorse_2,TechEndorse_3,TechEndorse_4,TechEndorse_5,TechEndorse_6,TechEndorse_7,TechEndorse_8,TechEndorse_9,TechEndorse_13,TechEndorse_13_TEXT,TechOppose_1,TechOppose_2,TechOppose_3,TechOppose_5,TechOppose_7,TechOppose_9,TechOppose_11,TechOppose_13,TechOppose_16,TechOppose_15,TechOppose_15_TEXT,Industry,JobSatPoints_1,JobSatPoints_2,JobSatPoints_3,JobSatPoints_4,JobSatPoints_5,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,JobSatPoints_13,JobSatPoints_14,JobSatPoints_15,JobSatPoints_16,JobSatPoints_15_TEXT,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,Currency,CompTotal,LanguageChoice,LanguageHaveWorkedWith,LanguageWantToWorkWith,LanguageAdmired,LanguagesHaveEntry,LanguagesWantEntry,DatabaseChoice,DatabaseHaveWorkedWith,DatabaseWantToWorkWith,DatabaseAdmired,DatabaseHaveEntry,DatabaseWantEntry,PlatformChoice,PlatformHaveWorkedWith,PlatformWantToWorkWith,PlatformAdmired,PlatformHaveEntry,PlatformWantEntry,WebframeChoice,WebframeHaveWorkedWith,WebframeWantToWorkWith,WebframeAdmired,WebframeHaveEntry,WebframeWantEntry,DevEnvsChoice,DevEnvsHaveWorkedWith,DevEnvsWantToWorkWith,DevEnvsAdmired,DevEnvHaveEntry,DevEnvWantEntry,SOTagsHaveWorkedWith,SOTagsWantToWorkWith,SOTagsAdmired,SOTagsHaveEntry,SOTagsWant Entry,OpSysPersonal use,OpSysProfessional use,OfficeStackAsyncHaveWorkedWith,OfficeStackAsyncWantToWorkWith,OfficeStackAsyncAdmired,OfficeStackHaveEntry,OfficeStackWantEntry,CommPlatformHaveWorkedWith,CommPlatformWantToWorkWith,CommPlatformAdmired,CommPlatformHaveEntr,CommPlatformWantEntr,AIModelsChoice,AIModelsHaveWorkedWith,AIModelsWantToWorkWith,AIModelsAdmired,AIModelsHaveEntry,AIModelsWantEntry,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SO_Dev_Content,SO_Actions_1,SO_Actions_16,SO_Actions_3,SO_Actions_4,SO_Actions_5,SO_Actions_6,SO_Actions_9,SO_Actions_7,SO_Actions_10,SO_Actions_15,SO_Actions_15_TEXT,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIToolCurrently partially AI,AIToolDon't plan to use AI for this task,AIToolPlan to partially use AI,AIToolPlan to mostly use AI,AIToolCurrently mostly AI,AIFrustration,AIExplain,AIAgents,AIAgentChange,AIAgent_Uses,AgentUsesGeneral,AIAgentImpactSomewhat agree,AIAgentImpactNeutral,AIAgentImpactSomewhat disagree,AIAgentImpactStrongly agree,AIAgentImpactStrongly disagree,AIAgentChallengesNeutral,AIAgentChallengesSomewhat disagree,AIAgentChallengesStrongly agree,AIAgentChallengesSomewhat agree,AIAgentChallengesStrongly disagree,AIAgentKnowledge,AIAgentKnowWrite,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",AI CodeGen tools or AI-enabled apps,14.0,"Developer, mobile",20 to 99 employees,People manager,Remote,"Yes, I influenced the purchase of a substantia...",Work,10.0,7.0,9.0,6.0,3.0,11.0,12.0,1.0,8.0,14.0,NaN,15.0,7.0,8.0,12.0,11.0,1.0,6.0,13.0,3.0,16.0,NaN,Fintech,3.0,1.0,4.0,9.0,5.0,10.0,12.0,11.0,2.0,6.0,7.0,13.0,14.0,15.0,8.0,NaN,I'm not sure,I have neither consider or transitioned into a...,7.0,3.0,Ukraine,EUR European Euro,52800.0,Yes,Bash/Shell (all shells);Dart;SQL,Dart,Dart,NaN,NaN,Yes,Cloud Firestore;PostgreSQL,NaN,NaN,NaN,NaN,Yes,Amazon Web Services (AWS);Cloudflare;Firebase;...,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,Yes,Android Studio;Notepad++;Visual Studio;Visual ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Windows;MacOS;Android,Windows;MacOS;Android;iOS;iPadOS,Confluence;GitHub;GitLab;Jira;Markdo

In [17]:
df_stack.info()

<class 'pandas.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: float64(52), int64(1), str(119)
memory usage: 64.6 MB


### 7.1 Selección de columnas tecnológicas

En esta parte se seleccionan solo las columnas necesarias para el análisis tecnológico.

Las columnas seleccionadas siguen principalmente dos patrones:

| Tipo de columna | Significado |
|---|---|
| `HaveWorkedWith` | Tecnologías que las personas encuestadas ya han usado. |
| `WantToWorkWith` | Tecnologías que las personas encuestadas quieren usar o aprender. |

Esta selección permite reducir el dataset original y centrarse solo en la información relevante para el análisis.

In [18]:
# ============================================================
# LIMPIEZA DE df_stack — SELECCIÓN DE COLUMNAS TECNOLÓGICAS
# ============================================================

columnas_stack_tech = [
    "ResponseId",

    # Lenguajes
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",

    # Bases de datos
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",

    # Plataformas / Cloud
    "PlatformHaveWorkedWith",
    "PlatformWantToWorkWith",

    # Frameworks web
    "WebframeHaveWorkedWith",
    "WebframeWantToWorkWith",

    # Entornos de desarrollo
    "DevEnvsHaveWorkedWith",
    "DevEnvsWantToWorkWith",

    # Modelos / herramientas IA
    "AIModelsHaveWorkedWith",
    "AIModelsWantToWorkWith"
]

# Nos quedamos solo con las columnas que existen realmente
columnas_stack_tech = [col for col in columnas_stack_tech if col in df_stack.columns]

df_stack_tech = df_stack[columnas_stack_tech].copy()

print("Columnas seleccionadas:")
print(columnas_stack_tech)

print("\nTamaño original de df_stack:", df_stack.shape)
print("Tamaño reducido de df_stack_tech:", df_stack_tech.shape)

df_stack_tech.head()

Columnas seleccionadas:
['ResponseId', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith']

Tamaño original de df_stack: (49191, 172)
Tamaño reducido de df_stack_tech: (49191, 13)


,ResponseId,LanguageHaveWorkedWith,LanguageWantToWorkWith,DatabaseHaveWorkedWith,DatabaseWantToWorkWith,PlatformHaveWorkedWith,PlatformWantToWorkWith,WebframeHaveWorkedWith,WebframeWantToWorkWith,DevEnvsHaveWorkedWith,DevEnvsWantToWorkWith,AIModelsHaveWorkedWith,AIModelsWantToWorkWith
0,1,Bash/Shell (all shells);Dart;SQL,Dart,Cloud Firestore;PostgreSQL,NaN,Amazon Web Services (AWS);Cloudflare;Firebase;...,NaN,NaN,NaN,Android Studio;Notepad++;Visual Studio;Visual ...,NaN,openAI GPT (chatbot models);openAI Image gener...,NaN
1,2,Java,Java;Python;Swift,Dynamodb;MongoDB,Dynamodb;MongoDB,Amazon Web Services (AWS);Datadog;Docker;Homeb...,Amazon Web Services (AWS);Datadog;Docker;Homeb...,Spring Boot,Spring Boot,IntelliJ IDEA;PyCharm;Visual Studio Code;Xcode,IntelliJ IDEA;PyCharm;Visual Studio Code;Xcode,openAI GPT (chatbot models),openAI GPT (chatbot models)
2,3,Dart;HTML/CSS;JavaScript;TypeScript,Dart;HTML/CSS;JavaScript;TypeScript,MongoDB;MySQL;PostgreSQL,Firebase Realtime Database;MongoDB,Datadog;Firebase;npm;pnpm,Amazon Web Services (AWS);Firebase;npm;pnpm;Vite,Next.js;Node.js;React,Next.js;Node.js;React,Visual Studio Code,Visual Studio Code;Windsurf,Gemini (Flash general purpose models);openAI G...,Gemini (Flash general purpose models);Gemini (...
3,4,Java;Kotlin;SQL,Java;Kotlin,NaN,NaN,Amazon Web Services (AWS);Google Cloud,Amazon Web Services (AWS);Google Cloud,Spring Boot,Spring Boot,NaN,NaN,NaN,NaN
4,5,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,C#;Java;JavaScript;Python;SQL;TypeScript,Elasticsearch;Microsoft SQL Server;MySQL;Oracl...,Elasticsearch;Oracle;PostgreSQL,Amazon Web Services (AWS);APT;Docker;Make;Mave...,Amazon Web Services (AWS);APT;Docker;Maven (bu...,Angular;ASP.NET;ASP.NET Core;Flask;jQuery,Angular;ASP.NET Core;Flask,Eclipse;IntelliJ IDEA;Jupyter Notebook/Jupyter...,IntelliJ IDEA;Jupyter Notebook/JupyterLab;Note...,openAI GPT (chatbot models),openAI GPT (chatbot models)


In [19]:
# Nulos en columnas tecnológicas seleccionadas
nulos_stack_tech = (
    df_stack_tech
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

nulos_stack_tech

AIModelsWantToWorkWith    37346
AIModelsHaveWorkedWith    32910
WebframeWantToWorkWith    31552
PlatformWantToWorkWith    29776
DatabaseWantToWorkWith    29458
DevEnvsWantToWorkWith     28577
WebframeHaveWorkedWith    26199
PlatformHaveWorkedWith    24933
DatabaseHaveWorkedWith    23641
DevEnvsHaveWorkedWith     23172
LanguageWantToWorkWith    22112
LanguageHaveWorkedWith    17520
ResponseId                    0
dtype: int64

In [20]:
# Porcentaje de nulos
pct_nulos_stack_tech = (
    df_stack_tech
    .isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

pct_nulos_stack_tech

AIModelsWantToWorkWith    75.92
AIModelsHaveWorkedWith    66.90
WebframeWantToWorkWith    64.14
PlatformWantToWorkWith    60.53
DatabaseWantToWorkWith    59.88
DevEnvsWantToWorkWith     58.09
WebframeHaveWorkedWith    53.26
PlatformHaveWorkedWith    50.69
DatabaseHaveWorkedWith    48.06
DevEnvsHaveWorkedWith     47.11
LanguageWantToWorkWith    44.95
LanguageHaveWorkedWith    35.62
ResponseId                 0.00
dtype: float64

In [21]:
# Reemplazar strings vacíos por NaN
df_stack_tech = df_stack_tech.replace("", np.nan)

df_stack_tech.head()

,ResponseId,LanguageHaveWorkedWith,LanguageWantToWorkWith,DatabaseHaveWorkedWith,DatabaseWantToWorkWith,PlatformHaveWorkedWith,PlatformWantToWorkWith,WebframeHaveWorkedWith,WebframeWantToWorkWith,DevEnvsHaveWorkedWith,DevEnvsWantToWorkWith,AIModelsHaveWorkedWith,AIModelsWantToWorkWith
0,1,Bash/Shell (all shells);Dart;SQL,Dart,Cloud Firestore;PostgreSQL,NaN,Amazon Web Services (AWS);Cloudflare;Firebase;...,NaN,NaN,NaN,Android Studio;Notepad++;Visual Studio;Visual ...,NaN,openAI GPT (chatbot models);openAI Image gener...,NaN
1,2,Java,Java;Python;Swift,Dynamodb;MongoDB,Dynamodb;MongoDB,Amazon Web Services (AWS);Datadog;Docker;Homeb...,Amazon Web Services (AWS);Datadog;Docker;Homeb...,Spring Boot,Spring Boot,IntelliJ IDEA;PyCharm;Visual Studio Code;Xcode,IntelliJ IDEA;PyCharm;Visual Studio Code;Xcode,openAI GPT (chatbot models),openAI GPT (chatbot models)
2,3,Dart;HTML/CSS;JavaScript;TypeScript,Dart;HTML/CSS;JavaScript;TypeScript,MongoDB;MySQL;PostgreSQL,Firebase Realtime Database;MongoDB,Datadog;Firebase;npm;pnpm,Amazon Web Services (AWS);Firebase;npm;pnpm;Vite,Next.js;Node.js;React,Next.js;Node.js;React,Visual Studio Code,Visual Studio Code;Windsurf,Gemini (Flash general purpose models);openAI G...,Gemini (Flash general purpose models);Gemini (...
3,4,Java;Kotlin;SQL,Java;Kotlin,NaN,NaN,Amazon Web Services (AWS);Google Cloud,Amazon Web Services (AWS);Google Cloud,Spring Boot,Spring Boot,NaN,NaN,NaN,NaN
4,5,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,C#;Java;JavaScript;Python;SQL;TypeScript,Elasticsearch;Microsoft SQL Server;MySQL;Oracl...,Elasticsearch;Oracle;PostgreSQL,Amazon Web Services (AWS);APT;Docker;Make;Mave...,Amazon Web Services (AWS);APT;Docker;Maven (bu...,Angular;ASP.NET;ASP.NET Core;Flask;jQuery,Angular;ASP.NET Core;Flask,Eclipse;IntelliJ IDEA;Jupyter Notebook/Jupyter...,IntelliJ IDEA;Jupyter Notebook/JupyterLab;Note...,openAI GPT (chatbot models),openAI GPT (chatbot models)


### 7.2 Transformación de tecnologías a formato largo

En el dataset original, muchas tecnologías aparecen juntas dentro de una misma celda, separadas por punto y coma.

Por ejemplo:

`Python;SQL;JavaScript`

Para facilitar el análisis, se transforma esta estructura en un formato largo, donde cada tecnología aparece en una fila independiente.

Ejemplo:

| ResponseId | technology |
|---|---|
| 1 | Python |
| 1 | SQL |
| 1 | JavaScript |

Este formato permite contar tecnologías, crear rankings y generar visualizaciones de forma mucho más sencilla.

In [22]:
# ============================================================
# FUNCIÓN PARA SEPARAR TECNOLOGÍAS
# ============================================================

def limpiar_y_explotar_tecnologias(df, columna, categoria, tipo):
    """
    Convierte una columna donde hay varias tecnologías separadas por ;
    en una tabla con una tecnología por fila.

    Ejemplo:
    Python;SQL;JavaScript

    Se convierte en:
    Python
    SQL
    JavaScript
    """

    df_temp = df[["ResponseId", columna]].dropna().copy()

    df_temp[columna] = (
        df_temp[columna]
        .astype(str)
        .str.split(";")
    )

    df_temp = df_temp.explode(columna)

    df_temp[columna] = (
        df_temp[columna]
        .astype(str)
        .str.strip()
    )

    df_temp = df_temp[df_temp[columna] != ""]

    df_temp = df_temp.rename(columns={columna: "technology"})

    df_temp["category"] = categoria
    df_temp["type"] = tipo

    return df_temp

### 7.3 Creación del dataset limpio de tecnologías

Después de separar las tecnologías, se crea un dataset limpio llamado `df_tech_clean`.

Este dataset contiene cuatro columnas principales:

| Columna | Significado |
|---|---|
| `ResponseId` | Identificador de la persona encuestada. |
| `technology` | Tecnología mencionada. |
| `category` | Categoría de la tecnología, por ejemplo lenguaje, base de datos o framework. |
| `type` | Indica si la tecnología es usada o demandada. |

Este dataset es una de las salidas más importantes de la limpieza, ya que será utilizado para crear gráficos y análisis posteriores.

In [23]:
# ============================================================
# CREACIÓN DEL DATASET LIMPIO DE TECNOLOGÍAS
# ============================================================

tablas_tecnologias = []

mapa_columnas = {
    "LanguageHaveWorkedWith": ("Lenguaje", "usada"),
    "LanguageWantToWorkWith": ("Lenguaje", "demandada"),

    "DatabaseHaveWorkedWith": ("Base de datos", "usada"),
    "DatabaseWantToWorkWith": ("Base de datos", "demandada"),

    "PlatformHaveWorkedWith": ("Plataforma / Cloud", "usada"),
    "PlatformWantToWorkWith": ("Plataforma / Cloud", "demandada"),

    "WebframeHaveWorkedWith": ("Framework web", "usada"),
    "WebframeWantToWorkWith": ("Framework web", "demandada"),

    "DevEnvsHaveWorkedWith": ("Entorno desarrollo", "usada"),
    "DevEnvsWantToWorkWith": ("Entorno desarrollo", "demandada"),

    "AIModelsHaveWorkedWith": ("Modelo / herramienta IA", "usada"),
    "AIModelsWantToWorkWith": ("Modelo / herramienta IA", "demandada")
}

for columna, (categoria, tipo) in mapa_columnas.items():
    if columna in df_stack_tech.columns:
        tabla = limpiar_y_explotar_tecnologias(
            df_stack_tech,
            columna,
            categoria,
            tipo
        )
        tablas_tecnologias.append(tabla)

df_tech_clean = pd.concat(tablas_tecnologias, ignore_index=True)

print("Tamaño del dataset limpio de tecnologías:", df_tech_clean.shape)

df_tech_clean.head(20)

Tamaño del dataset limpio de tecnologías: (1176875, 4)


,ResponseId,technology,category,type
0,1,Bash/Shell (all shells),Lenguaje,usada
1,1,Dart,Lenguaje,usada
2,1,SQL,Lenguaje,usada
3,2,Java,Lenguaje,usada
4,3,Dart,Lenguaje,usada
5,3,HTML/CSS,Lenguaje,usada
6,3,JavaScript,Lenguaje,usada
7,3,TypeScript,Lenguaje,usada
8,4,Java,Lenguaje,usada
9,4,Kotlin,Lenguaje,usada


In [24]:
# Comprobaciones del dataset limpio de tecnologías

print("Tamaño:", df_tech_clean.shape)

print("\nValores por tipo:")
print(df_tech_clean["type"].value_counts())

print("\nValores por categoría:")
print(df_tech_clean["category"].value_counts())

print("\nPrimeras tecnologías:")
print(df_tech_clean["technology"].value_counts().head(20))

Tamaño: (1176875, 4)

Valores por tipo:
type
usada        696085
demandada    480790
Name: count, dtype: int64

Valores por categoría:
category
Lenguaje                   334363
Plataforma / Cloud         284352
Entorno desarrollo         157687
Base de datos              155719
Framework web              147237
Modelo / herramienta IA     97517
Name: count, dtype: int64

Primeras tecnologías:
technology
Visual Studio Code             32594
JavaScript                     31586
Python                         30829
HTML/CSS                       30359
SQL                            29890
Docker                         29657
PostgreSQL                     26392
Bash/Shell (all shells)        24165
TypeScript                     23958
openAI GPT (chatbot models)    21752
npm                            20411
Node.js                        18350
Amazon Web Services (AWS)      17721
React                          17620
SQLite                         16983
MySQL                          15701


## 8. Creación de rankings de tecnologías

A partir del dataset limpio `df_tech_clean`, se crean rankings de tecnologías.

Estos rankings agrupan los datos por:

- Categoría tecnológica.
- Tipo de tecnología: usada o demandada.
- Nombre de la tecnología.

El resultado permite saber cuántas veces aparece cada tecnología dentro de cada grupo.

Se generan tres tablas principales:

| Tabla | Descripción |
|---|---|
| `ranking_tecnologias` | Ranking completo de tecnologías por categoría y tipo. |
| `ranking_usadas` | Ranking filtrado solo con tecnologías usadas. |
| `ranking_demandadas` | Ranking filtrado solo con tecnologías demandadas o deseadas. |

In [25]:
# ============================================================
# CREACIÓN DE RANKINGS DE TECNOLOGÍAS
# ============================================================

ranking_tecnologias = (
    df_tech_clean
    .groupby(["category", "type", "technology"])
    .size()
    .reset_index(name="count")
    .sort_values(["category", "type", "count"], ascending=[True, True, False])
)

print("Tamaño del ranking:", ranking_tecnologias.shape)

ranking_tecnologias.head(30)

Tamaño del ranking: (372, 4)


,category,type,technology,count
24,Base de datos,demandada,PostgreSQL,11863
26,Base de datos,demandada,SQLite,7185
25,Base de datos,demandada,Redis,6014
20,Base de datos,demandada,MySQL,5120
19,Base de datos,demandada,MongoDB,4421
18,Base de datos,demandada,Microsoft SQL Server,3851
11,Base de datos,demandada,Elasticsearch,3288
16,Base de datos,demandada,MariaDB,3239
10,Base de datos,demandada,Dynamodb,1741
28,Base de datos,demandada,Supabase,1621


In [26]:
# Ver top 10 por categoría y tipo
for categoria in ranking_tecnologias["category"].unique():
    for tipo in ranking_tecnologias["type"].unique():
        print(f"\n{categoria.upper()} - {tipo.upper()}")
        display(
            ranking_tecnologias[
                (ranking_tecnologias["category"] == categoria) &
                (ranking_tecnologias["type"] == tipo)
            ].head(10)
        )


BASE DE DATOS - DEMANDADA


,category,type,technology,count
24,Base de datos,demandada,PostgreSQL,11863
26,Base de datos,demandada,SQLite,7185
25,Base de datos,demandada,Redis,6014
20,Base de datos,demandada,MySQL,5120
19,Base de datos,demandada,MongoDB,4421
18,Base de datos,demandada,Microsoft SQL Server,3851
11,Base de datos,demandada,Elasticsearch,3288
16,Base de datos,demandada,MariaDB,3239
10,Base de datos,demandada,Dynamodb,1741
28,Base de datos,demandada,Supabase,1621



BASE DE DATOS - USADA


,category,type,technology,count
54,Base de datos,usada,PostgreSQL,14529
50,Base de datos,usada,MySQL,10581
56,Base de datos,usada,SQLite,9798
48,Base de datos,usada,Microsoft SQL Server,7871
55,Base de datos,usada,Redis,7316
49,Base de datos,usada,MongoDB,6267
46,Base de datos,usada,MariaDB,5862
41,Base de datos,usada,Elasticsearch,4347
52,Base de datos,usada,Oracle,2761
40,Base de datos,usada,Dynamodb,2551



ENTORNO DESARROLLO - DEMANDADA


...

,category,type,technology,count
285,Modelo / herramienta IA,usada,openAI GPT (chatbot models),13424
273,Modelo / herramienta IA,usada,Anthropic: Claude Sonnet,7063
277,Modelo / herramienta IA,usada,Gemini (Flash general purpose models),5823
287,Modelo / herramienta IA,usada,openAI Reasoning models,5716
286,Modelo / herramienta IA,usada,openAI Image generating models,4395
278,Modelo / herramienta IA,usada,Gemini (Pro Reasoning models),4221
275,Modelo / herramienta IA,usada,DeepSeek (R- Reasoning models),3848
279,Modelo / herramienta IA,usada,Meta Llama (all models),2941
276,Modelo / herramienta IA,usada,DeepSeek (V- General purpose models),2363
284,Modelo / herramienta IA,usada,X Grok models,1839



PLATAFORMA / CLOUD - DEMANDADA


,category,type,technology,count
298,Plataforma / Cloud,demandada,Docker,12243
289,Plataforma / Cloud,demandada,Amazon Web Services (AWS),7103
305,Plataforma / Cloud,demandada,Kubernetes,6772
328,Plataforma / Cloud,demandada,npm,6505
315,Plataforma / Cloud,demandada,Pip,4743
324,Plataforma / Cloud,demandada,Vite,4458
309,Plataforma / Cloud,demandada,Microsoft Azure,4089
300,Plataforma / Cloud,demandada,Google Cloud,3995
294,Plataforma / Cloud,demandada,Cloudflare,3844
322,Plataforma / Cloud,demandada,Terraform,3817



PLATAFORMA / CLOUD - USADA


,category,type,technology,count
340,Plataforma / Cloud,usada,Docker,17414
370,Plataforma / Cloud,usada,npm,13906
331,Plataforma / Cloud,usada,Amazon Web Services (AWS),10618
357,Plataforma / Cloud,usada,Pip,10025
347,Plataforma / Cloud,usada,Kubernetes,6993
351,Plataforma / Cloud,usada,Microsoft Azure,6447
345,Plataforma / Cloud,usada,Homebrew,6291
366,Plataforma / Cloud,usada,Vite,6215
342,Plataforma / Cloud,usada,Google Cloud,6015
349,Plataforma / Cloud,usada,Make,5682


In [27]:
# ============================================================
# SEPARAR RANKINGS DE USADAS Y DEMANDADAS
# ============================================================

ranking_usadas = ranking_tecnologias[
    ranking_tecnologias["type"] == "usada"
].copy()

ranking_demandadas = ranking_tecnologias[
    ranking_tecnologias["type"] == "demandada"
].copy()

print("Ranking usadas:", ranking_usadas.shape)
print("Ranking demandadas:", ranking_demandadas.shape)

display(ranking_usadas.head(10))
display(ranking_demandadas.head(10))

Ranking usadas: (186, 4)
Ranking demandadas: (186, 4)


,category,type,technology,count
54,Base de datos,usada,PostgreSQL,14529
50,Base de datos,usada,MySQL,10581
56,Base de datos,usada,SQLite,9798
48,Base de datos,usada,Microsoft SQL Server,7871
55,Base de datos,usada,Redis,7316
49,Base de datos,usada,MongoDB,6267
46,Base de datos,usada,MariaDB,5862
41,Base de datos,usada,Elasticsearch,4347
52,Base de datos,usada,Oracle,2761
40,Base de datos,usada,Dynamodb,2551


,category,type,technology,count
24,Base de datos,demandada,PostgreSQL,11863
26,Base de datos,demandada,SQLite,7185
25,Base de datos,demandada,Redis,6014
20,Base de datos,demandada,MySQL,5120
19,Base de datos,demandada,MongoDB,4421
18,Base de datos,demandada,Microsoft SQL Server,3851
11,Base de datos,demandada,Elasticsearch,3288
16,Base de datos,demandada,MariaDB,3239
10,Base de datos,demandada,Dynamodb,1741
28,Base de datos,demandada,Supabase,1621


## 9. Guardado de archivos limpios

En esta sección se guardan los datasets resultantes de la limpieza en la carpeta `data/clean/`.

Los archivos generados son:

| Archivo | Descripción |
|---|---|
| `jobs_clean.csv` | Dataset limpio de ofertas de empleo. |
| `tecno_jobs_clean.csv` | Dataset limpio de ofertas tecnológicas. |
| `stack_tech_columns_clean.csv` | Dataset reducido de Stack Overflow con columnas tecnológicas. |
| `technologies_clean_long_format.csv` | Dataset limpio en formato largo, con una tecnología por fila. |
| `technology_rankings.csv` | Ranking completo de tecnologías. |
| `technology_rankings_used.csv` | Ranking de tecnologías usadas. |
| `technology_rankings_wanted.csv` | Ranking de tecnologías demandadas. |

Estos archivos quedan preparados para que el resto del equipo pueda utilizarlos en el análisis exploratorio y en la visualización de datos.

In [28]:
# ============================================================
# GUARDAR ARCHIVOS LIMPIOS
# ============================================================

os.makedirs("data/clean", exist_ok=True)

df_jobs_clean.to_csv("data/clean/jobs_clean.csv", index=False)
df_tecno_clean.to_csv("data/clean/tecno_jobs_clean.csv", index=False)

df_stack_tech.to_csv("data/clean/stack_tech_columns_clean.csv", index=False)
df_tech_clean.to_csv("data/clean/technologies_clean_long_format.csv", index=False)

ranking_tecnologias.to_csv("data/clean/technology_rankings.csv", index=False)
ranking_usadas.to_csv("data/clean/technology_rankings_used.csv", index=False)
ranking_demandadas.to_csv("data/clean/technology_rankings_wanted.csv", index=False)

print("Archivos guardados correctamente en data/clean/")

Archivos guardados correctamente en data/clean/


In [29]:
os.listdir("data/clean")

['jobs_all_clean.csv',
 'jobs_clean.csv',
 'scraping_jobs_template.csv',
 'stack_tech_columns_clean.csv',
 'technologies_clean_long_format.csv',
 'technology_rankings.csv',
 'technology_rankings_used.csv',
 'technology_rankings_wanted.csv',
 'tecno_jobs_clean.csv']

## 10.Unificación de datasets de ofertas de empleo

Después de limpiar los tres datasets principales, se decide no unirlos todos en un único archivo, ya que no todos representan el mismo tipo de información.

Los datasets `df_jobs` y `df_tecno` sí pueden unificarse porque ambos tienen la misma unidad de análisis: **una fila representa una oferta de empleo**.

Sin embargo, el dataset `df_stack` no se une con ellos porque representa respuestas de personas encuestadas y tecnologías utilizadas o deseadas. Por tanto, se mantiene separado para el análisis tecnológico.

| Dataset limpio | Unidad de análisis | Información que aporta | Uso en el proyecto |
|---|---|---|---|
| `jobs_clean.csv` | Una fila = una oferta de empleo | Puesto, empresa, ubicación, salario, industria, seniority y skills solicitadas | Analizar ofertas laborales y habilidades pedidas en el mercado |
| `tecno_jobs_clean.csv` | Una fila = una oferta tecnológica scrapeada | Título, empresa, salario, ubicación, tipo de trabajo, fecha de publicación y enlace | Complementar el análisis de ofertas tecnológicas reales |
| `jobs_all_clean.csv` | Una fila = una oferta de empleo unificada | Unión de `jobs_clean.csv` y `tecno_jobs_clean.csv` con columnas comunes | Dataset preparado para analizar ofertas de empleo de forma conjunta |
| `technologies_clean_long_format.csv` | Una fila = una tecnología mencionada por una persona | Tecnología, categoría tecnológica y si es usada o demandada | Analizar tecnologías usadas y demandadas por profesionales |
| `technology_rankings.csv` | Una fila = una tecnología agrupada | Ranking de tecnologías por categoría y tipo | Crear gráficos y comparativas de tecnologías más usadas y demandadas |

### Decisión de limpieza

Se crea un dataset unificado solo para las ofertas de empleo:

`jobs_all_clean.csv`

Este archivo combina la información de `df_jobs_clean` y `df_tecno_clean`, dejando columnas homogéneas para facilitar el análisis posterior.

El dataset de Stack Overflow se mantiene separado porque responde a otra pregunta de análisis: **qué tecnologías usan y quieren usar los profesionales**.

In [30]:
# ============================================================
# UNIFICACIÓN DE DATASETS DE OFERTAS DE EMPLEO
# ============================================================

# Creamos una versión estandarizada de df_jobs_clean
jobs_1 = pd.DataFrame()

jobs_1["job_title"] = df_jobs_clean.get("job_title")
jobs_1["company"] = df_jobs_clean.get("company")
jobs_1["location"] = df_jobs_clean.get("location")
jobs_1["salary"] = df_jobs_clean.get("salary")
jobs_1["job_type"] = np.nan
jobs_1["post_date"] = df_jobs_clean.get("post_date")
jobs_1["link"] = np.nan
jobs_1["skills"] = df_jobs_clean.get("skills")
jobs_1["industry"] = df_jobs_clean.get("industry")
jobs_1["seniority_level"] = df_jobs_clean.get("seniority_level")
jobs_1["source_dataset"] = "df_jobs"


# Creamos una versión estandarizada de df_tecno_clean
jobs_2 = pd.DataFrame()

jobs_2["job_title"] = df_tecno_clean.get("titulo")
jobs_2["company"] = df_tecno_clean.get("empresa")
jobs_2["location"] = df_tecno_clean.get("ubicacion")
jobs_2["salary"] = df_tecno_clean.get("salario")
jobs_2["job_type"] = df_tecno_clean.get("tipo_de_trabajo")
jobs_2["post_date"] = df_tecno_clean.get("fecha_de_publicacion")
jobs_2["link"] = df_tecno_clean.get("enlace")
jobs_2["skills"] = np.nan
jobs_2["industry"] = np.nan
jobs_2["seniority_level"] = np.nan
jobs_2["source_dataset"] = "df_tecno"


# Unimos ambos datasets
jobs_all_clean = pd.concat([jobs_1, jobs_2], ignore_index=True)

# Eliminamos filas completamente vacías en las columnas principales
columnas_principales = ["job_title", "company", "location", "salary", "job_type", "post_date", "link"]

jobs_all_clean = jobs_all_clean.dropna(
    how="all",
    subset=columnas_principales
)

# Eliminamos duplicados exactos
duplicados_jobs_all = jobs_all_clean.duplicated().sum()
print(f"Duplicados encontrados en jobs_all_clean: {duplicados_jobs_all}")

jobs_all_clean = jobs_all_clean.drop_duplicates()

# Comprobación final
print("Tamaño de df_jobs_clean:", df_jobs_clean.shape)
print("Tamaño de df_tecno_clean:", df_tecno_clean.shape)
print("Tamaño de jobs_all_clean:", jobs_all_clean.shape)

jobs_all_clean.head()

Duplicados encontrados en jobs_all_clean: 2
Tamaño de df_jobs_clean: (944, 13)
Tamaño de df_tecno_clean: (600, 7)
Tamaño de jobs_all_clean: (1542, 11)


,job_title,company,location,salary,job_type,post_date,link,skills,industry,seniority_level,source_dataset
0,data scientist,company_003,"Grapevine, TX . Hybrid","€100,472 - €200,938",NaN,17 days ago,NaN,"['spark', 'r', 'python', 'scala', 'machine lea...",Retail,senior,df_jobs
1,data scientist,company_005,"Fort Worth, TX . Hybrid","€118,733",NaN,15 days ago,NaN,"['spark', 'r', 'python', 'sql', 'machine learn...",Manufacturing,lead,df_jobs
2,data scientist,company_007,"Austin, TX . Toronto, Ontario, Canada . Kirkla...","€94,987 - €159,559",NaN,a month ago,NaN,"['aws', 'git', 'python', 'docker', 'sql', 'mac...",Technology,senior,df_jobs
3,data scientist,company_008,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...","€112,797 - €194,402",NaN,8 days ago,NaN,"['sql', 'r', 'python']",Technology,senior,df_jobs
4,data scientist,company_009,On-site,"€114,172 - €228,337",NaN,3 days ago,NaN,[],Finance,NaN,df_jobs


In [31]:
# Guardar dataset unificado de ofertas
jobs_all_clean.to_csv("data/clean/jobs_all_clean.csv", index=False)

print("Archivo jobs_all_clean.csv guardado correctamente en data/clean/")

Archivo jobs_all_clean.csv guardado correctamente en data/clean/


In [32]:
os.listdir("data/clean")

['jobs_all_clean.csv',
 'jobs_clean.csv',
 'scraping_jobs_template.csv',
 'stack_tech_columns_clean.csv',
 'technologies_clean_long_format.csv',
 'technology_rankings.csv',
 'technology_rankings_used.csv',
 'technology_rankings_wanted.csv',
 'tecno_jobs_clean.csv']

## 11.Plantilla para nuevos datos de scraping

Para facilitar la integración de futuros datos obtenidos mediante scraping, se define una estructura estándar de columnas.

El objetivo es que cualquier nuevo dataset de ofertas de empleo pueda adaptarse fácilmente al formato de `jobs_all_clean.csv`.

De esta forma, si otro miembro del equipo extrae nuevas ofertas desde una web, podrá guardar los datos siguiendo esta plantilla y después unirlos al dataset limpio de ofertas.

| Columna esperada | Descripción |
|---|---|
| `job_title` | Título del puesto u oferta de empleo. |
| `company` | Nombre de la empresa. |
| `location` | Ubicación de la oferta. |
| `salary` | Salario publicado, si está disponible. |
| `job_type` | Tipo de trabajo: remoto, híbrido, presencial, jornada completa, etc. |
| `post_date` | Fecha de publicación de la oferta. |
| `link` | Enlace a la oferta original. |
| `skills` | Habilidades o tecnologías mencionadas en la oferta. |
| `industry` | Industria o sector de la empresa, si está disponible. |
| `seniority_level` | Nivel de experiencia requerido, si aparece. |
| `source_dataset` | Fuente de procedencia del dato. |

Esta plantilla permite mantener una estructura homogénea entre los datasets de ofertas de empleo y facilita el análisis posterior.

In [33]:
# ============================================================
# PLANTILLA PARA NUEVOS DATOS DE SCRAPING
# ============================================================

scraping_template_columns = [
    "job_title",
    "company",
    "location",
    "salary",
    "job_type",
    "post_date",
    "link",
    "skills",
    "industry",
    "seniority_level",
    "source_dataset"
]

scraping_template = pd.DataFrame(columns=scraping_template_columns)

scraping_template.to_csv("data/clean/scraping_jobs_template.csv", index=False)

print("Plantilla scraping_jobs_template.csv guardada correctamente en data/clean/")

Plantilla scraping_jobs_template.csv guardada correctamente en data/clean/


## 12.Limpieza extra de salarios

En esta sección se realiza una limpieza básica de las columnas relacionadas con salarios.

Los salarios pueden aparecer en distintos formatos: texto, rangos, símbolos de moneda, valores vacíos o información no numérica.  
Por este motivo, se crea una función para extraer valores numéricos aproximados y generar columnas auxiliares que faciliten el análisis posterior.

Esta limpieza no pretende calcular salarios exactos en todos los casos, sino dejar una versión más estructurada y comparable de la información salarial disponible.

In [40]:
# ============================================================
# LIMPIEZA EXTRA DE SALARIOS - VERSIÓN MEJORADA
# ============================================================

def limpiar_salario_a_numero(valor):
    """
    Extrae un salario numérico aproximado.

    Casos soportados:
    - "$50,000 - $70,000"      -> 60000
    - "€30K - €40K"            -> 35000
    - "30k-45k"                -> 37500
    - "50000"                  -> 50000
    - "From 60,000 per year"   -> 60000
    - "Not specified"          -> NaN

    Si encuentra varios valores, devuelve la media.
    Si encuentra un solo valor, devuelve ese valor.
    """

    if pd.isna(valor):
        return np.nan

    texto = str(valor).lower().strip()

    valores_invalidos = [
        "",
        "nan",
        "none",
        "null",
        "not specified",
        "not provided",
        "unknown",
        "sin especificar",
        "no especificado",
        "not disclosed",
        "competitive",
        "competitive salary"
    ]

    if texto in valores_invalidos:
        return np.nan

    # Normalizar separadores
    texto = texto.replace(",", "")
    texto = texto.replace("€", "")
    texto = texto.replace("$", "")
    texto = texto.replace("£", "")
    texto = texto.replace("usd", "")
    texto = texto.replace("eur", "")
    texto = texto.replace("gbp", "")

    # Buscar números con posible decimal y posible K
    # Ejemplos que captura: 30k, 45.5k, 50000
    coincidencias = re.findall(r"\d+(?:\.\d+)?\s*k?|\d+(?:\.\d+)?", texto)

    if len(coincidencias) == 0:
        return np.nan

    numeros = []

    for item in coincidencias:
        item = item.strip()

        if "k" in item:
            numero = float(item.replace("k", "").strip()) * 1000
        else:
            numero = float(item)

        numeros.append(numero)

    if len(numeros) == 0:
        return np.nan

    return np.mean(numeros)

In [35]:
# Aplicar limpieza de salarios al dataset unificado de ofertas

if "salary" in jobs_all_clean.columns:
    jobs_all_clean["salary_clean"] = jobs_all_clean["salary"].apply(limpiar_salario_a_numero)

    print("Columna salary_clean creada correctamente.")
    print(jobs_all_clean[["salary", "salary_clean"]].head(20))
else:
    print("No existe la columna salary en jobs_all_clean.")

Columna salary_clean creada correctamente.
                 salary  salary_clean
0   €100,472 - €200,938      150705.0
1              €118,733      118733.0
2    €94,987 - €159,559      127273.0
3   €112,797 - €194,402      153599.5
4   €114,172 - €228,337      171254.5
5   €196,371 - €251,170      223770.5
6     €51,330 - €70,144       60737.0
7   €121,480 - €132,440      126960.0
8              €207,331      207331.0
9              €219,201      219201.0
10              €96,815       96815.0
11    €62,697 - €89,293       75995.0
12   €99,464 - €165,954      132709.0
13  €112,343 - €117,823      115083.0
14  €127,867 - €150,700      139283.5
15             €134,266      134266.0
16              €52,746       52746.0
17  €126,956 - €198,196      162576.0
18  €185,412 - €228,338      206875.0
19              €80,123       80123.0


In [42]:
# Resumen de salary_clean

if "salary_clean" in jobs_all_clean.columns:
    print("Valores no nulos en salary_clean:", jobs_all_clean["salary_clean"].notnull().sum())
    print("Valores nulos en salary_clean:", jobs_all_clean["salary_clean"].isnull().sum())

    display(jobs_all_clean["salary_clean"].describe())

Valores no nulos en salary_clean: 1071
Valores nulos en salary_clean: 471


count    1.071000e+03
mean     1.204878e+05
std      1.248324e+05
min      7.055000e+03
25%      4.596825e+04
50%      1.193995e+05
75%      1.657745e+05
max      2.739979e+06
Name: salary_clean, dtype: float64

In [43]:
# Guardar jobs_all_clean actualizado con salary_clean

jobs_all_clean.to_csv("data/clean/jobs_all_clean.csv", index=False)

print("jobs_all_clean.csv actualizado con la columna salary_clean.")

jobs_all_clean.csv actualizado con la columna salary_clean.


## 13.Limpieza extra de ubicaciones y ciudades

En esta sección se realiza una limpieza básica de la columna `location` del dataset unificado de ofertas de empleo.

Las ubicaciones pueden aparecer en distintos formatos, por ejemplo:

- `Madrid, Spain`
- `Barcelona`
- `Remote`
- `Hybrid - Madrid`
- `Spain`
- valores vacíos o no especificados

El objetivo es crear columnas auxiliares que faciliten el análisis posterior:

| Columna | Descripción |
|---|---|
| `location_clean` | Ubicación normalizada en texto limpio. |
| `city_clean` | Ciudad principal extraída de la ubicación. |
| `is_remote` | Indica si la oferta parece ser remota. |

Esta limpieza es aproximada, pero permite mejorar el análisis de distribución geográfica de las ofertas.

In [44]:
# ============================================================
# LIMPIEZA EXTRA DE UBICACIONES / CIUDADES
# ============================================================

def limpiar_ubicacion(valor):
    """
    Limpia una ubicación:
    - elimina espacios extra
    - convierte valores vacíos o no informativos en NaN
    - normaliza algunos textos comunes
    """

    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()

    valores_invalidos = [
        "",
        "nan",
        "none",
        "null",
        "not specified",
        "not provided",
        "unknown",
        "sin especificar",
        "no especificado"
    ]

    if texto.lower() in valores_invalidos:
        return np.nan

    # Eliminar espacios repetidos
    texto = re.sub(r"\s+", " ", texto)

    return texto


def extraer_ciudad(valor):
    """
    Extrae una ciudad aproximada desde una ubicación.

    Ejemplos:
    - "Madrid, Spain" -> "Madrid"
    - "Barcelona, Catalonia, Spain" -> "Barcelona"
    - "Hybrid - Madrid" -> "Madrid"
    - "Remote" -> "Remote"
    """

    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()

    # Detectar remoto
    if "remote" in texto.lower() or "remoto" in texto.lower():
        return "Remote"

    # Quitar prefijos comunes
    texto = re.sub(r"(?i)^hybrid\s*[-:]\s*", "", texto)
    texto = re.sub(r"(?i)^híbrido\s*[-:]\s*", "", texto)
    texto = re.sub(r"(?i)^onsite\s*[-:]\s*", "", texto)
    texto = re.sub(r"(?i)^presencial\s*[-:]\s*", "", texto)

    # Si hay comas, nos quedamos con la primera parte
    if "," in texto:
        ciudad = texto.split(",")[0].strip()
    else:
        ciudad = texto.strip()

    if ciudad == "":
        return np.nan

    return ciudad


def detectar_remoto(valor):
    """
    Detecta si una ubicación parece indicar trabajo remoto.
    """

    if pd.isna(valor):
        return False

    texto = str(valor).lower()

    palabras_remoto = [
        "remote",
        "remoto",
        "work from home",
        "wfh",
        "home office"
    ]

    return any(palabra in texto for palabra in palabras_remoto)

In [45]:
# Aplicar limpieza de ubicaciones al dataset unificado de ofertas

if "location" in jobs_all_clean.columns:
    jobs_all_clean["location_clean"] = jobs_all_clean["location"].apply(limpiar_ubicacion)
    jobs_all_clean["city_clean"] = jobs_all_clean["location_clean"].apply(extraer_ciudad)
    jobs_all_clean["is_remote"] = jobs_all_clean["location_clean"].apply(detectar_remoto)

    print("Columnas location_clean, city_clean e is_remote creadas correctamente.")
    display(jobs_all_clean[["location", "location_clean", "city_clean", "is_remote"]].head(30))
else:
    print("No existe la columna location en jobs_all_clean.")

Columnas location_clean, city_clean e is_remote creadas correctamente.


,location,location_clean,city_clean,is_remote
0,"Grapevine, TX . Hybrid","Grapevine, TX . Hybrid",Grapevine,False
1,"Fort Worth, TX . Hybrid","Fort Worth, TX . Hybrid",Fort Worth,False
2,"Austin, TX . Toronto, Ontario, Canada . Kirkla...","Austin, TX . Toronto, Ontario, Canada . Kirkla...",Austin,False
3,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...","Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...",Chicago,False
4,On-site,On-site,On-site,False
5,"New York, NY","New York, NY",New York,False
6,"Berkeley, CA","Berkeley, CA",Berkeley,False
7,"Menlo Park, CA","Menlo Park, CA",Menlo Park,False
8,Fully Remote,Fully Remote,Remote,True
9,On-site,On-site,On-site,False


In [46]:
# Comprobación de ubicaciones limpias

if "city_clean" in jobs_all_clean.columns:
    print("Top 20 ciudades / ubicaciones limpias:")
    display(jobs_all_clean["city_clean"].value_counts(dropna=False).head(20))

if "is_remote" in jobs_all_clean.columns:
    print("\nDistribución de ofertas remotas:")
    display(jobs_all_clean["is_remote"].value_counts())

Top 20 ciudades / ubicaciones limpias:


city_clean
Madrid           242
NaN              183
Barcelona        108
Remote            79
Bengaluru         67
New York          50
San Francisco     42
Mountain View     29
United States     23
Málaga            22
Seattle           19
On-site           18
Toronto           18
Chicago           17
Valencia          16
San Jose          15
Sunnyvale         15
San Diego         14
McLean            14
Boston            13
Name: count, dtype: int64


Distribución de ofertas remotas:


is_remote
False    1463
True       79
Name: count, dtype: int64

In [47]:
# Guardar jobs_all_clean actualizado con columnas de ubicación

jobs_all_clean.to_csv("data/clean/jobs_all_clean.csv", index=False)

print("jobs_all_clean.csv actualizado con columnas de ubicación limpias.")

jobs_all_clean.csv actualizado con columnas de ubicación limpias.


## 14.Detección inicial de outliers

En esta sección se realiza una detección inicial de posibles valores atípicos en las columnas numéricas generadas durante la limpieza.

El análisis se centra principalmente en `salary_clean`, ya que es la variable numérica más relevante dentro del dataset unificado de ofertas de empleo.

Para detectar posibles outliers se utiliza el método del rango intercuartílico, también conocido como método IQR.

Este método calcula:

- Q1: primer cuartil.
- Q3: tercer cuartil.
- IQR: rango intercuartílico.
- Límite inferior: Q1 - 1.5 × IQR.
- Límite superior: Q3 + 1.5 × IQR.

Los valores que quedan fuera de estos límites se marcan como posibles outliers.

Esta detección no elimina automáticamente los datos, solo crea una columna auxiliar para identificar posibles valores extremos y facilitar el análisis posterior.

In [48]:
# ============================================================
# DETECCIÓN INICIAL DE OUTLIERS
# ============================================================

def detectar_outliers_iqr(df, columna):
    """
    Detecta outliers en una columna numérica usando el método IQR.

    Devuelve:
    - DataFrame con una nueva columna booleana: nombre_columna_outlier
    - Diccionario con Q1, Q3, IQR y límites
    """

    df = df.copy()

    if columna not in df.columns:
        print(f"La columna {columna} no existe en el DataFrame.")
        return df, None

    serie = df[columna].dropna()

    if serie.empty:
        print(f"La columna {columna} no tiene valores válidos.")
        return df, None

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    columna_outlier = f"{columna}_outlier"

    df[columna_outlier] = (
        (df[columna] < limite_inferior) |
        (df[columna] > limite_superior)
    )

    resumen = {
        "columna": columna,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "n_outliers": df[columna_outlier].sum(),
        "pct_outliers": round(df[columna_outlier].mean() * 100, 2)
    }

    return df, resumen

In [49]:
# Aplicar detección de outliers sobre salary_clean

jobs_all_clean, resumen_outliers_salary = detectar_outliers_iqr(
    jobs_all_clean,
    "salary_clean"
)

resumen_outliers_salary

{'columna': 'salary_clean',
 'q1': np.float64(45968.25),
 'q3': np.float64(165774.5),
 'iqr': np.float64(119806.25),
 'limite_inferior': np.float64(-133741.125),
 'limite_superior': np.float64(345483.875),
 'n_outliers': np.int64(3),
 'pct_outliers': np.float64(0.19)}

In [50]:
# Ver posibles outliers salariales

if "salary_clean_outlier" in jobs_all_clean.columns:
    outliers_salary = jobs_all_clean[
        jobs_all_clean["salary_clean_outlier"] == True
    ].copy()

    print("Número de posibles outliers salariales:", outliers_salary.shape[0])

    display(
        outliers_salary[
            ["job_title", "company", "location", "salary", "salary_clean", "source_dataset"]
        ].head(20)
    )

Número de posibles outliers salariales: 3


,job_title,company,location,salary,salary_clean,source_dataset
108,data scientist,company_209,"Bengaluru, Karnataka, India","€639,348",639348.0,df_jobs
536,data scientist,company_967,Greater Bengaluru Area . Hybrid,"€2,739,979",2739979.0,df_jobs
702,data scientist,company_967,Greater Hyderabad Area,"€2,283,322",2283322.0,df_jobs


In [51]:
# Guardar jobs_all_clean actualizado con columna de outliers

jobs_all_clean.to_csv("data/clean/jobs_all_clean.csv", index=False)

print("jobs_all_clean.csv actualizado con detección inicial de outliers.")

jobs_all_clean.csv actualizado con detección inicial de outliers.


## 15.Comprobación final de archivos generados

En esta sección se comprueba que todos los archivos limpios se han generado correctamente en la carpeta `data/clean/`.

También se revisa el tamaño de los principales datasets resultantes para confirmar que el proceso de limpieza, transformación y guardado se ha ejecutado correctamente.

In [52]:
# ============================================================
# COMPROBACIÓN FINAL DE ARCHIVOS GENERADOS
# ============================================================

archivos_generados = os.listdir("data/clean")

print("Archivos generados en data/clean/:")

for archivo in archivos_generados:
    print("-", archivo)

Archivos generados en data/clean/:
- jobs_all_clean.csv
- jobs_clean.csv
- scraping_jobs_template.csv
- stack_tech_columns_clean.csv
- technologies_clean_long_format.csv
- technology_rankings.csv
- technology_rankings_used.csv
- technology_rankings_wanted.csv
- tecno_jobs_clean.csv


In [53]:
# ============================================================
# COMPROBACIÓN DE TAMAÑOS FINALES
# ============================================================

print("Tamaños finales de los datasets limpios:\n")

print("jobs_clean:", df_jobs_clean.shape)
print("tecno_jobs_clean:", df_tecno_clean.shape)
print("jobs_all_clean:", jobs_all_clean.shape)
print("stack_tech_columns_clean:", df_stack_tech.shape)
print("technologies_clean_long_format:", df_tech_clean.shape)
print("technology_rankings:", ranking_tecnologias.shape)
print("technology_rankings_used:", ranking_usadas.shape)
print("technology_rankings_wanted:", ranking_demandadas.shape)

Tamaños finales de los datasets limpios:

jobs_clean: (944, 13)
tecno_jobs_clean: (600, 7)
jobs_all_clean: (1542, 16)
stack_tech_columns_clean: (49191, 13)
technologies_clean_long_format: (1176875, 4)
technology_rankings: (372, 4)
technology_rankings_used: (186, 4)
technology_rankings_wanted: (186, 4)


In [54]:
# Comprobación rápida de lectura del CSV final unificado

jobs_all_check = pd.read_csv("data/clean/jobs_all_clean.csv")

print("Archivo jobs_all_clean.csv leído correctamente.")
print("Tamaño:", jobs_all_check.shape)

jobs_all_check.head()

Archivo jobs_all_clean.csv leído correctamente.
Tamaño: (1542, 16)


,job_title,company,location,salary,job_type,post_date,link,skills,industry,seniority_level,source_dataset,salary_clean,location_clean,city_clean,is_remote,salary_clean_outlier
0,data scientist,company_003,"Grapevine, TX . Hybrid","€100,472 - €200,938",NaN,17 days ago,NaN,"['spark', 'r', 'python', 'scala', 'machine lea...",Retail,senior,df_jobs,150705.0,"Grapevine, TX . Hybrid",Grapevine,False,False
1,data scientist,company_005,"Fort Worth, TX . Hybrid","€118,733",NaN,15 days ago,NaN,"['spark', 'r', 'python', 'sql', 'machine learn...",Manufacturing,lead,df_jobs,118733.0,"Fort Worth, TX . Hybrid",Fort Worth,False,False
2,data scientist,company_007,"Austin, TX . Toronto, Ontario, Canada . Kirkla...","€94,987 - €159,559",NaN,a month ago,NaN,"['aws', 'git', 'python', 'docker', 'sql', 'mac...",Technology,senior,df_jobs,127273.0,"Austin, TX . Toronto, Ontario, Canada . Kirkla...",Austin,False,False
3,data scientist,company_008,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...","€112,797 - €194,402",NaN,8 days ago,NaN,"['sql', 'r', 'python']",Technology,senior,df_jobs,153599.5,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...",Chicago,False,False
4,data scientist,company_009,On-site,"€114,172 - €228,337",NaN,3 days ago,NaN,[],Finance,NaN,df_jobs,171254.5,On-site,On-site,False,False


## 16.Conclusión

En este notebook se ha realizado la limpieza y preparación de los tres datasets principales del proyecto.

El proceso ha incluido:

- Normalización de nombres de columnas.
- Limpieza básica de textos.
- Eliminación de duplicados.
- Revisión y tratamiento inicial de valores nulos.
- Selección de columnas relevantes para el análisis.
- Transformación del dataset tecnológico a formato largo.
- Creación de rankings de tecnologías usadas y demandadas.
- Unificación de los datasets de ofertas de empleo compatibles.
- Creación de una plantilla para futuros datos obtenidos mediante scraping.
- Limpieza básica de salarios.
- Limpieza básica de ubicaciones y ciudades.
- Detección inicial de posibles outliers salariales.
- Guardado de archivos limpios para el resto del equipo.

Como resultado, se han generado datasets limpios en la carpeta `data/clean/`, preparados para continuar con las siguientes fases del proyecto: análisis exploratorio, visualización de datos y extracción de conclusiones.

Los archivos principales generados son:

| Archivo | Finalidad |
|---|---|
| `jobs_clean.csv` | Dataset limpio de ofertas de empleo. |
| `tecno_jobs_clean.csv` | Dataset limpio de ofertas tecnológicas. |
| `jobs_all_clean.csv` | Dataset unificado de ofertas de empleo, incluyendo columnas auxiliares como `salary_clean`, `location_clean`, `city_clean`, `is_remote` y `salary_clean_outlier`. |
| `stack_tech_columns_clean.csv` | Dataset reducido con columnas tecnológicas de Stack Overflow. |
| `technologies_clean_long_format.csv` | Dataset en formato largo con una tecnología por fila. |
| `technology_rankings.csv` | Ranking completo de tecnologías usadas y demandadas. |
| `technology_rankings_used.csv` | Ranking de tecnologías usadas. |
| `technology_rankings_wanted.csv` | Ranking de tecnologías demandadas. |
| `scraping_jobs_template.csv` | Plantilla de columnas para integrar futuros datos obtenidos mediante scraping. |

El dataset `jobs_all_clean.csv` queda preparado para analizar ofertas laborales de forma conjunta, mientras que los archivos relacionados con tecnologías permiten estudiar qué herramientas son más usadas y cuáles son más demandadas por los profesionales.

No se ha realizado un modelo relacional completo ni un `JOIN` entre todos los datasets, ya que no existe una clave común fiable entre ellos. La unificación realizada ha sido una concatenación vertical con `pandas`, aplicada únicamente a los datasets de ofertas de empleo porque comparten la misma unidad de análisis: una fila representa una oferta.